# requirements-to-code: train and evaluate on Colab

Fine-tunes Qwen2.5-0.5B with LoRA using this repo's hand-written training loop (`train.py`), then measures
**pass@1** for the base and fine-tuned models on `eval_problems.json` using `evaluate.py`.

**Before running:** `Runtime -> Change runtime type -> GPU` (T4 is fine; L4/A100 is faster).

The notebook imports the repo's own modules and calls them directly, so what runs here is exactly the code in
the repo. Settings come from `config.py`; you can override them in section 4 without editing files.

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Use Runtime -> Change runtime type -> GPU."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
major, minor = torch.cuda.get_device_capability(0)
print(f"compute capability {major}.{minor}",
      "-> native bfloat16" if major >= 8 else
      "-> no native bfloat16 (e.g. T4): bf16 still runs but is emulated and slower")

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
torch 2.11.0+cu130 | CUDA 13.0 | Tesla T4
compute capability 7.5 -> no native bfloat16 (e.g. T4): bf16 still runs but is emulated and slower


## 2. Clone the repo

In [2]:
import os

REPO_URL = "https://github.com/SanketJadhav7d3/requirement-to-code.git"
BRANCH = "main"
REPO_DIR = "/content/requirement-to-code"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1

Already up to date.
/content/requirement-to-code
1acd482 (HEAD -> main, origin/main, origin/HEAD) Expand eval set to 60 problems and add Colab notebook


## 3. Install dependencies

Colab already ships a CUDA build of PyTorch; this adds `peft`, `bitsandbytes` and friends.

In [3]:
!pip install -q -r requirements.txt

import sys, importlib
sys.path.insert(0, REPO_DIR)   # make sure the repo's evaluate.py wins over any pip package named `evaluate`
for m in ["transformers", "peft", "datasets", "accelerate", "bitsandbytes"]:
    try:
        print(f"{m:13s}", importlib.import_module(m).__version__)
    except ImportError:
        print(f"{m:13s} not installed")

transformers  5.17.0
peft          0.21.0
datasets      4.8.5
accelerate    1.15.0
bitsandbytes  0.50.2


In [4]:
print('Upgrading torchao...')
!pip install torchao --upgrade

Upgrading torchao...


## 4. Configure the run

`config.py` holds the defaults. `train.py`, `data.py` and `evaluate.py` all share the same `cfg` object, so
changing it here changes the run.

- `SMOKE_TEST = True`: 200 examples, 1 epoch. Use it to check everything works in a few minutes.
- `SAVE_TO_DRIVE = True`: writes the adapter and results to Google Drive so they survive the session ending.

In [5]:
SMOKE_TEST = False
SAVE_TO_DRIVE = False

from config import cfg

if SMOKE_TEST:
    cfg.train_size = 200
    cfg.epochs = 1

# Other overrides, e.g.:
# cfg.use_qlora = True
# cfg.target_modules = ("q_proj", "k_proj", "v_proj", "o_proj")

OUT_DIR = REPO_DIR
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/requirement-to-code"
    os.makedirs(OUT_DIR, exist_ok=True)
    cfg.adapter_dir = os.path.join(OUT_DIR, "adapter")

cfg

Config(model_name='Qwen/Qwen2.5-0.5B', max_len=512, dataset='iamtarun/python_code_instructions_18k_alpaca', train_size=2000, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, target_modules=('q_proj', 'v_proj'), use_qlora=False, epochs=3, batch_size=4, grad_accum=2, lr=0.0002, warmup_ratio=0.03, log_every=20, adapter_dir='adapter', eval_file='eval_problems.json', max_new_tokens=256, subprocess_timeout=10)

## 5. Load the eval problems

In [6]:
import json
from collections import Counter

with open(cfg.eval_file, encoding="utf-8") as f:
    problems = json.load(f)
print(len(problems), "problems")
print(", ".join(p["id"] for p in problems))

60 problems
fizzbuzz, is_palindrome, two_sum, flatten, word_count, roman_to_int, merge_intervals, gcd, reverse_string, factorial, is_prime, fibonacci, sum_list, max_subarray, count_vowels, capitalize_words, remove_duplicates, binary_search, is_anagram, celsius_to_fahrenheit, second_largest, char_frequency, rotate_list, sum_digits, is_even, find_max, count_words, average, square_list, filter_evens, is_leap_year, reverse_words, power_of_two, list_product, lcm, count_occurrences, run_length_encode, valid_parentheses, chunk_list, int_to_binary, caesar_cipher, merge_sorted, missing_number, transpose, group_anagrams, longest_common_prefix, digital_root, is_sorted, longest_unique_substring, int_to_roman, climb_stairs, coin_change, lis_length, edit_distance, primes_up_to, permutations, matrix_multiply, spiral_order, top_k_frequent, balanced_split


## 6. Baseline: evaluate the untuned model

Each generation is run with its asserts in a separate Python process with a timeout (`evaluate.run_in_subprocess`).

In [7]:
import evaluate as ev
assert ev.__file__.startswith(REPO_DIR), f"wrong module imported: {ev.__file__}"

base_passed, base_rows = ev.evaluate("base", problems)
print(f"\nbase: {base_passed}/{len(problems)} pass@1 = {base_passed / len(problems):.0%}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[base] fizzbuzz         FAIL  (AssertionError)
[base] is_palindrome    FAIL  (AssertionError)
[base] two_sum          PASS
[base] flatten          FAIL  (SyntaxError: '[' was never closed)
[base] word_count       FAIL  (AssertionError)
[base] roman_to_int     FAIL  (NameError: name 'roman_numer' is not defined. Did you mean: 'roman_numerals'?)
[base] merge_intervals  FAIL  (SyntaxError: '[' was never closed)
[base] gcd              PASS
[base] reverse_string   PASS
[base] factorial        PASS
[base] is_prime         PASS
[base] fibonacci        PASS
[base] sum_list         PASS
[base] max_subarray     FAIL  (SyntaxError: '[' was never closed)
[base] count_vowels     FAIL  (SyntaxError: '(' was never closed)
[base] capitalize_words PASS
[base] remove_duplicates FAIL  (SyntaxError: '[' was never closed)
[base] binary_search    PASS
[base] is_anagram       PASS
[base] celsius_to_fahrenheit PASS
[base] second_largest   FAIL  (AssertionError)
[base] char_frequency   PASS
[base] rotate_list

## 7. Fine-tune with LoRA

Runs `train.main()`: the hand-written loop (forward -> scaled loss -> backward -> clip -> AdamW step ->
scheduler step -> zero_grad). The loss should fall over the run. It saves only the LoRA adapter to `cfg.adapter_dir`.

## 8. Evaluate the fine-tuned model

In [9]:
ft_passed, ft_rows = ev.evaluate("finetuned", problems)
print(f"\nfinetuned: {ft_passed}/{len(problems)} pass@1 = {ft_passed / len(problems):.0%}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at 'adapter'

## 9. Compare

In [ ]:
import pandas as pd

n = len(problems)
df = pd.DataFrame({
    "id": [r[0] for r in base_rows],
    "base": [r[1] for r in base_rows],
    "finetuned": [r[1] for r in ft_rows],
    "base_error": [r[2] for r in base_rows],
    "finetuned_error": [r[2] for r in ft_rows],
})

print("| Model      | pass@1          |")
print("|------------|-----------------|")
print(f"| base       | {base_passed}/{n} ({base_passed / n:.0%}) |")
print(f"| finetuned  | {ft_passed}/{n} ({ft_passed / n:.0%}) |")

fixed = df[~df.base & df.finetuned].id.tolist()
broke = df[df.base & ~df.finetuned].id.tolist()
both_fail = df[~df.base & ~df.finetuned].id.tolist()
print("\nfixed by fine-tune:", fixed or "-")
print("regressed:         ", broke or "-")
print("fail in both:      ", both_fail or "-")

def err_type(e):
    return e.split(":")[0] if e else "PASS"

print("\nerror types  base:", dict(Counter(map(err_type, df.base_error))))
print("error types  ft:  ", dict(Counter(map(err_type, df.finetuned_error))))

df

## 10. Save results

Writes `eval_results.csv` and `eval_summary.json` next to the adapter. Copy the numbers into `results.md`,
including the training time, which is still blank there.

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "eval_results.csv"), index=False)
summary = {
    "model": cfg.model_name,
    "gpu": torch.cuda.get_device_name(0),
    "train_size": cfg.train_size,
    "epochs": cfg.epochs,
    "lora_r": cfg.lora_r,
    "lora_alpha": cfg.lora_alpha,
    "target_modules": list(cfg.target_modules),
    "use_qlora": cfg.use_qlora,
    "train_minutes": round(train_minutes, 1),
    "n_problems": n,
    "base_pass": base_passed,
    "finetuned_pass": ft_passed,
}
with open(os.path.join(OUT_DIR, "eval_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
summary

Optional: download the adapter and results as a zip (not needed if you saved to Drive).

In [ ]:
import shutil
from google.colab import files

stage = "/content/run_outputs"
shutil.copytree(cfg.adapter_dir, os.path.join(stage, "adapter"), dirs_exist_ok=True)
for fn in ["eval_results.csv", "eval_summary.json"]:
    shutil.copy(os.path.join(OUT_DIR, fn), stage)
bundle = shutil.make_archive(stage, "zip", stage)
files.download(bundle)